In [4]:
import pandas as pd

# Define the path to your LSOA lookup file
full_lookup_path = '/Users/callumhall/Documents/Dissertation/Datasets/PCD to LSOA to MSOA to LAD.csv'

# THE FIX: Add the encoding='latin1' parameter
df_lookup = pd.read_csv(full_lookup_path, encoding='latin1')

# Verify the file loaded correctly
df_lookup.info()

/var/folders/_k/7j49pfp16td9jhxfyktpm0xh0000gn/T/ipykernel_33946/292748039.py:7: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df_lookup = pd.read_csv(full_lookup_path, encoding='latin1')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2668999 entries, 0 to 2668998
Data columns (total 14 columns):
 #   Column    Dtype  
---  ------    -----  
 0   pcd7      object 
 1   pcd8      object 
 2   pcds      object 
 3   dointr    int64  
 4   doterm    float64
 5   usertype  int64  
 6   oa11cd    object 
 7   lsoa11cd  object 
 8   msoa11cd  object 
 9   ladcd     object 
 10  lsoa11nm  object 
 11  msoa11nm  object 
 12  ladnm     object 
 13  ladnmw    object 
dtypes: float64(1), int64(2), object(11)
memory usage: 285.1+ MB


In [6]:
df_lsoa_lookup = pd.read_csv('/Users/callumhall/Documents/Dissertation/Datasets/LSOA 11 to LSOA 21 to LAD22.csv')
df_lsoa_lookup.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34753 entries, 0 to 34752
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   ObjectId  34753 non-null  int64 
 1   LSOA11CD  34753 non-null  object
 2   LSOA11NM  34753 non-null  object
 3   LSOA21CD  34753 non-null  object
 4   LSOA21NM  34753 non-null  object
 5   LAD22CD   34753 non-null  object
 6   LAD22NM   34753 non-null  object
 7   LAD22NMW  1909 non-null   object
dtypes: int64(1), object(7)
memory usage: 2.1+ MB


In [8]:
import pandas as pd

# --- 1. Define paths to your two lookup files ---
path_postcode_lookup = '/Users/callumhall/Documents/Dissertation/Datasets/PCD to LSOA to MSOA to LAD.csv'
path_lsoa_bridge = '/Users/callumhall/Documents/Dissertation/Datasets/LSOA 11 to LSOA 21 to LAD22.csv'

# --- 2. Load both lookup files with the correct columns ---
# Load the 2011-based postcode lookup
df_lookup_2011 = pd.read_csv(
    path_postcode_lookup, 
    encoding='latin1', 
    usecols=['pcds', 'lsoa11cd', 'msoa11cd', 'ladcd', 'ladnm']
)

# Load the bridge file connecting 2011 to 2021 geographies
# The column names are taken directly from the info you provided
df_bridge = pd.read_csv(
    path_lsoa_bridge,
    usecols=['LSOA11CD', 'LSOA21CD', 'LAD22CD', 'LAD22NM']
)

# --- 3. Merge the two lookups to create the master file ---
# We join them using the 2011 LSOA code, which is the common link.
df_master_lookup = pd.merge(
    df_lookup_2011, 
    df_bridge, 
    left_on='lsoa11cd', 
    right_on='LSOA11CD', 
    how='inner'
)

# --- 4. Clean and save the new master lookup ---
# Drop the redundant 2011 LSOA code from the bridge file
df_master_lookup = df_master_lookup.drop(columns=['LSOA11CD'])

# Save the final file
output_path = 'master_lookup_2021_based.csv'
df_master_lookup.to_csv(output_path, index=False)

print(f"Created a new, corrected lookup file named: {output_path}")
print("\n--- Info for your new master lookup file ---")
df_master_lookup.info()

Created a new, corrected lookup file named: master_lookup_2021_based.csv

--- Info for your new master lookup file ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2359627 entries, 0 to 2359626
Data columns (total 8 columns):
 #   Column    Dtype 
---  ------    ----- 
 0   pcds      object
 1   lsoa11cd  object
 2   msoa11cd  object
 3   ladcd     object
 4   ladnm     object
 5   LSOA21CD  object
 6   LAD22CD   object
 7   LAD22NM   object
dtypes: object(8)
memory usage: 144.0+ MB
